In [3]:
pip install bleak


   ------------------------------------ ---  9/10 [bleak]
   ---------------------------------------- 10/10 [bleak]

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [6]:
import asyncio
import json
import os
import socket
from bleak import BleakScanner

DB_FILE = "registered_users.json"

def load_users():
    if os.path.exists(DB_FILE):
        with open(DB_FILE, "r") as f:
            return json.load(f)
    return {}

def save_users(users):
    with open(DB_FILE, "w") as f:
        json.dump(users, f, indent=4)

async def sign_up():
    print("\n--- SIGN UP ---")
    print("Scanning for 15 seconds... (Ensure your phone's Bluetooth settings menu is open!)")
    
    devices = await BleakScanner.discover(timeout=15.0)
    named_devices = [d for d in devices if d.name]
    
    if not named_devices:
        print("❌ No discoverable devices found.")
        return
        
    print("\nAvailable Devices:")
    for i, d in enumerate(named_devices):
        print(f"[{i}] {d.name} (MAC: {d.address})")
        
    try:
        choice = input("\nEnter device number (or 'q' to quit): ")
        if choice.lower() == 'q': return
        
        choice_idx = int(choice)
        if 0 <= choice_idx < len(named_devices):
            selected = named_devices[choice_idx]
            username = input(f"Enter player username for '{selected.name}': ")
            
            users = load_users()
            users[selected.address] = {"username": username, "device_name": selected.name}
            save_users(users)
            
            print(f"\n✅ Success! '{username}' registered to '{selected.name}'.")
        else:
            print("❌ Invalid selection.")
    except ValueError:
        print("❌ Invalid input.")

async def sign_in():
    print("\n--- SIGN IN ---")
    users = load_users()
    if not users:
        print("❌ No users registered yet. Please sign up a device first.")
        return None
        
    print("Continuously scanning for registered devices... (Interrupt kernel to stop)")
    
    attempt = 1
    while True:
        devices = await BleakScanner.discover(timeout=10.0)
        for d in devices:
            if d.address in users:
                username = users[d.address]["username"]
                print(f"\n✅ SIGN IN SUCCESSFUL!")
                print(f"Welcome back, {username}! (Detected via {d.name})")
                return username
                
        print(f"Scan {attempt} complete. Not found yet. Rescanning...")
        attempt += 1
        await asyncio.sleep(1)

async def main_menu(conn):
    while True:
        print("\n===============================")
        print("  BLUETOOTH LOGIN SYSTEM")
        print("===============================")
        print("1. Sign Up")
        print("2. Sign In")
        print("3. Exit")
        
        choice = input("Select an option (1-3): ")
        
        if choice == '1':
            await sign_up()
        elif choice == '2':
            username = await sign_in()
            if username:
                # SEND THE DATA OVER THE SOCKET!
                msg = bytes(f"LOGIN:{username}", "utf-8")
                conn.send(msg)
                print(f"\n📡 Sent login data for '{username}' to C# Game!")
        elif choice == '3':
            # Tell C# to close the connection cleanly
            conn.send(bytes("q", "utf-8"))
            conn.close()
            print("Exiting...")
            break
        else:
            print("Invalid choice.")

# --- 1. START THE SOCKET SERVER ---
mySocket = socket.socket()
mySocket.bind(("localhost", 5000))
mySocket.listen(1)

print("⏳ Waiting for C# Game to connect on port 5000...")
print("   --> Start your C# Game NOW <--")

# The cell will PAUSE here until you start the C# program
conn, addr = mySocket.accept()
print(f"✅ C# Game Connected from {addr}!")

# --- 2. START THE BLUETOOTH MENU ---
await main_menu(conn)

⏳ Waiting for C# Game to connect on port 5000...
   --> Start your C# Game NOW <--
✅ C# Game Connected from ('127.0.0.1', 6900)!

  BLUETOOTH LOGIN SYSTEM
1. Sign Up
2. Sign In
3. Exit

--- SIGN UP ---
Scanning for 15 seconds... (Ensure your phone's Bluetooth settings menu is open!)
❌ No discoverable devices found.

  BLUETOOTH LOGIN SYSTEM
1. Sign Up
2. Sign In
3. Exit

--- SIGN UP ---
Scanning for 15 seconds... (Ensure your phone's Bluetooth settings menu is open!)

Available Devices:
[0] Buds Pro (MAC: C4:D5:60:AE:DE:73)
[1] Galaxy Buds Pro (6FB9) LE (MAC: C4:17:CD:AA:92:44)

✅ Success! '0' registered to 'Galaxy Buds Pro (6FB9) LE'.

  BLUETOOTH LOGIN SYSTEM
1. Sign Up
2. Sign In
3. Exit

--- SIGN IN ---
Continuously scanning for registered devices... (Interrupt kernel to stop)

✅ SIGN IN SUCCESSFUL!
Welcome back, 0! (Detected via Galaxy Buds Pro (6FB9) LE)

📡 Sent login data for '0' to C# Game!

  BLUETOOTH LOGIN SYSTEM
1. Sign Up
2. Sign In
3. Exit


ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host